## 步驟 1：載入 OpenAI API 金鑰

**說明：** 我們需要 OpenAI 的 API 來把文字轉成向量。

- `%load_ext dotenv`：載入環境變數工具
- `%dotenv`：從檔案讀取你的 API 金鑰（避免把金鑰直接寫在程式碼裡）

In [2]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [3]:
import os
from openai import OpenAI

# 建立 OpenAI 客戶端（會自動讀取 API 金鑰）
client = OpenAI()

## 步驟 2：準備範例資料

我們準備了 21 句名言，分成三個主題：
- 🕊️ **自由（Freedom）**
- 🤝 **友誼（Friendship）**
- 🍕 **食物（Food）**

**目標：** 等一下我們會看看 AI 能不能自動把相同主題的句子找出來！

In [4]:
phrases = [
    # 關於自由的名言
    "Freedom consists not in doing what we like, but in having the right to do what we ought.",
    # 翻譯：自由不在於做我們想做的事，而在於有權利做我們應該做的事。
    
    "Those who deny freedom to others deserve it not for themselves.",
    # 翻譯：拒絕給他人自由的人，自己也不配擁有自由。
    
    "Liberty, when it begins to take root, is a plant of rapid growth.",
    # 翻譯：自由一旦開始生根，就會快速成長。
    
    "Freedom lies in being bold.",
    # 翻譯：自由在於勇敢。
    
    "Is freedom anything else than the right to live as we wish?",
    # 翻譯：自由不就是按照我們的意願生活的權利嗎？
    
    "I am no bird and no net ensnares me: I am a free human being with an independent will.",
    # 翻譯：我不是鳥，沒有網能困住我：我是有獨立意志的自由人。
    
    "The secret to happiness is freedom... And the secret to freedom is courage."
    # 翻譯：幸福的秘訣是自由...而自由的秘訣是勇氣。
    
    "Freedom is the oxygen of the soul.", 
    # 翻譯：自由是靈魂的氧氣。
    
    "Life without liberty is like a body without spirit."
    # 翻譯：沒有自由的生活就像沒有靈魂的身體。
    
    # 關於友誼的名言
    "There is nothing on this earth more to be prized than true friendship.",
    # 翻譯：世上沒有比真正的友誼更珍貴的東西。
    
    "There are no strangers here; Only friends you haven't yet met.",
    # 翻譯：這裡沒有陌生人；只有你還沒遇到的朋友。
    
    "Friendship is the only cement that will ever hold the world together.",
    # 翻譯：友誼是唯一能把世界凝聚在一起的黏著劑。
    
    "A true friend is someone who is there for you when he'd rather be anywhere else.",
    # 翻譯：真正的朋友是即使他寧願在別處，也會陪在你身邊的人。
    
    "Friendship is the golden thread that ties the heart of all the world.", 
    # 翻譯：友誼是連結全世界心靈的金色絲線。
    
    "Your friend is the man who knows all about you and still likes you.",
    # 翻譯：你的朋友是那個了解你的一切還喜歡你的人。
    
    "A single rose can be my garden... a single friend, my world."
    # 翻譯：一朵玫瑰可以是我的花園...一個朋友，就是我的世界。
    
    # 關於食物的名言
    "One cannot think well, love well, sleep well, if one has not dined well.",
    # 翻譯：如果沒有吃好，就無法好好思考、好好愛、好好睡。
    
    "Let food be thy medicine and medicine be thy food.",
    # 翻譯：讓食物成為你的藥物，讓藥物成為你的食物。
    
    "People who love to eat are always the best people.",
    # 翻譯：喜歡吃的人總是最棒的人。
    
    "The only way to get rid of a temptation is to yield to it.",
    # 翻譯：擺脫誘惑的唯一方法就是屈服於它。（隱喻美食的誘惑）
    
    "Food is our common ground, a universal experience.",
    # 翻譯：食物是我們的共同點，是普世的經驗。
    
    "Life is uncertain. Eat dessert first.",
    # 翻譯：人生無常，先吃甜點。
    
    "All you need is love. But a little chocolate now and then doesn't hurt."
    # 翻譯：你需要的只是愛。但偶爾來點巧克力也無妨。
]

In [5]:
# 看看總共有幾句話
print(f"總共有 {len(phrases)} 句名言")
print(f"\n第一句話是：{phrases[0]}")

總共有 20 句名言

第一句話是：Freedom consists not in doing what we like, but in having the right to do what we ought.


## 步驟 3：把第一句話轉成向量（單一輸入範例）

**重點說明：**
- 使用 `text-embedding-3-small` 模型
- 這個模型會把文字轉成 **1536 個數字**（1536 維向量）
- 每個數字代表文字在某個「概念維度」上的特徵

**類比：** 就像你用身高、體重、年齡來描述一個人，AI 用 1536 個數字來描述一段文字的「意思」。

In [6]:
# 取第一句話
first_phrase = phrases[0]
print(f"要轉換的句子：{first_phrase}")

要轉換的句子：Freedom consists not in doing what we like, but in having the right to do what we ought.


In [7]:
# 呼叫 OpenAI API 來產生向量（embedding）
response = client.embeddings.create(
    input = first_phrase,           # 輸入：要轉換的文字
    model = "text-embedding-3-small" # 模型：OpenAI 的嵌入模型
)

# 看看回傳的結果
print("API 回傳的資料結構：")
print(response.data)

API 回傳的資料結構：
[Embedding(embedding=[0.027506759390234947, 0.019747251644730568, 0.03171176090836525, 0.0784779042005539, 0.08465763181447983, -0.04685906693339348, -0.009008231572806835, 0.028505738824605942, -0.006975426338613033, -0.00541597418487072, 0.023266907781362534, -0.017249805852770805, -0.01807454228401184, 0.006992850452661514, 0.03777533024549484, 0.007997636683285236, 0.004385051317512989, -0.026484549045562744, 0.018086159601807594, 0.052550919353961945, 0.04281668737530708, 0.0024016143288463354, -0.007381987292319536, 0.02381286211311817, -0.04590655118227005, -0.006232000421732664, 0.019793715327978134, -0.003441249020397663, -0.028807755559682846, 0.021803289651870728, 0.033756185322999954, -0.008613286539912224, -0.020130580291152, 0.03308245539665222, 0.045976247638463974, 0.02525324933230877, 0.016053354367613792, -0.05775490403175354, 0.032594580203294754, 0.024858305230736732, -0.009885242208838463, -0.06175081804394722, -0.0005528504261747003, 0.052876170724630

In [8]:
# 提取向量（embedding）
first_embedding = response.data[0].embedding

print(f"向量的維度（長度）：{len(first_embedding)}")
print(f"\n向量的前 10 個數字：{first_embedding[:10]}")
print("\n💡 這 1536 個數字就代表這句話的「意思」！")

向量的維度（長度）：1536

向量的前 10 個數字：[0.027506759390234947, 0.019747251644730568, 0.03171176090836525, 0.0784779042005539, 0.08465763181447983, -0.04685906693339348, -0.009008231572806835, 0.028505738824605942, -0.006975426338613033, -0.00541597418487072]

💡 這 1536 個數字就代表這句話的「意思」！


## 步驟 4：把所有句子都轉成向量（批次處理）

### 方法 1：用迴圈逐一轉換（比較慢）

**說明：** 一次處理一句話，適合學習理解流程。

In [9]:
# 先定義一個輔助函式
def get_embedding(text, model="text-embedding-3-small"):
    """
    把一段文字轉成向量
    
    參數：
        text: 要轉換的文字
        model: 使用的 AI 模型
    
    回傳：
        一個包含 1536 個數字的列表（向量）
    """
    # 移除換行符號（清理文字）
    text = text.replace("\n", " ")
    
    # 呼叫 API
    response = client.embeddings.create(input=[text], model=model)
    
    # 回傳向量
    return response.data[0].embedding

In [10]:
# 方法 1：用 for 迴圈（容易理解）
embeddings = []

for doc in phrases:
    doc_emb = get_embedding(doc)  # 把每句話轉成向量
    embeddings.append(doc_emb)    # 加到列表裡

print(f"✅ 完成！總共產生了 {len(embeddings)} 個向量")
print(f"每個向量有 {len(embeddings[0])} 個數字")

✅ 完成！總共產生了 20 個向量
每個向量有 1536 個數字


In [11]:
# 方法 2：用 List Comprehension（Python 的簡潔寫法）
# 這行程式碼的效果跟上面的 for 迴圈完全一樣！
embeddings = [get_embedding(doc) for doc in phrases]

print("✅ 用簡潔語法也完成了！")

✅ 用簡潔語法也完成了！


### 方法 3：一次傳送所有文字（最有效率）

**優點：**
- 只需要呼叫 API 一次（省時間、省錢）
- OpenAI API 支援批次處理

**建議：** 實際專案中優先使用這個方法！

In [12]:
# 一次傳送所有句子
response = client.embeddings.create(
    input = phrases,                  # 直接傳整個列表！
    model = "text-embedding-3-small"
)

# 提取所有向量
embeddings = [item.embedding for item in response.data]

print(f"✅ 批次處理完成！一次產生 {len(embeddings)} 個向量")

✅ 批次處理完成！一次產生 20 個向量


In [13]:
# 可以轉成 NumPy 陣列，方便後續數學運算
import numpy as np

embeddings_array = np.array(embeddings)
print(f"陣列形狀：{embeddings_array.shape}")  # (21, 1536) = 21 句話，每句 1536 維
print(f"\n💡 意思是：21 句話 × 1536 個數字 = 完整的向量矩陣")

陣列形狀：(20, 1536)

💡 意思是：21 句話 × 1536 個數字 = 完整的向量矩陣


## 步驟 5：使用向量資料庫（ChromaDB）

### 為什麼需要向量資料庫？

雖然我們可以手動計算向量之間的相似度，但當資料量很大（成千上萬筆）時，會遇到問題：
- ❌ 計算太慢
- ❌ 記憶體不夠
- ❌ 每次都要重新計算

**向量資料庫的優勢：**
- ✅ 快速搜尋（使用特殊索引技術）
- ✅ 自動計算相似度
- ✅ 可以儲存文字 + 向量 + 其他資訊

### ChromaDB 簡介

- 輕量級向量資料庫
- 適合學習和小型專案
- 可以在記憶體中運行（不用安裝資料庫伺服器）

### 📚 其他常見的向量資料庫：
- **Pinecone**：雲端服務，適合大型專案
- **Weaviate**：開源，功能豐富
- **FAISS**：Facebook 開發，超快速度
- **Milvus**：適合大規模部署

![ChromaDB 架構](img/02_chroma.png)

### 在記憶體中運行 ChromaDB（最簡單的方式）

**優點：**
- 不需要安裝額外軟體
- 立即開始使用

**缺點：**
- 程式結束後資料就消失了
- 不適合大量資料

In [14]:
import chromadb

# 建立 ChromaDB 客戶端（在記憶體中運行）
chroma_client = chromadb.Client()

print("✅ ChromaDB 客戶端已建立！")

✅ ChromaDB 客戶端已建立！


### 建立集合（Collection）

**概念類比：**
- 資料庫 = 整個圖書館
- 集合（Collection）= 一個書架
- 文件（Document）= 一本書

一個集合可以存放很多相關的文件和向量。

In [15]:
# 建立一個叫 "nice_phrases" 的集合
collection = chroma_client.create_collection(name = "nice_phrases")

print("✅ 集合已建立：nice_phrases")

✅ 集合已建立：nice_phrases


### 把資料加入集合

每筆資料需要三個部分：
1. **ID**：唯一識別碼（例如：id0, id1, id2...）
2. **Document**：原始文字
3. **Embedding**：向量（1536 個數字）

In [16]:
# 準備 ID（每句話一個唯一編號）
ids = [f"id{i}" for i in range(len(phrases))]

print(f"ID 範例：{ids[:5]}...")
print(f"總共 {len(ids)} 個 ID")

ID 範例：['id0', 'id1', 'id2', 'id3', 'id4']...
總共 20 個 ID


In [17]:
# 把所有資料加入集合
collection.add(
    embeddings = embeddings,  # 向量列表
    documents = phrases,      # 原始文字列表
    ids = ids                 # ID 列表
)

print("✅ 所有資料已加入資料庫！")
print(f"   - {len(phrases)} 句名言")
print(f"   - {len(embeddings)} 個向量")
print(f"   - {len(ids)} 個 ID")

✅ 所有資料已加入資料庫！
   - 20 句名言
   - 20 個向量
   - 20 個 ID


## 步驟 6：語意搜尋（Semantic Search）

### 什麼是語意搜尋？

**傳統關鍵字搜尋：**
- 搜尋「朋友」→ 只找到包含「朋友」這個詞的句子
- 找不到「友誼」、"friendship" 等相關內容

**語意搜尋：**
- 搜尋「朋友」→ 找到所有關於友誼、朋友、友情的句子
- AI 理解「意思」，不只是文字

### 原理：
1. 把你的問題轉成向量
2. 跟資料庫裡的所有向量比較
3. 找出最相似的幾筆

**相似度計算：** 常用「餘弦相似度」（Cosine Similarity）
- 值越接近 0 = 越相似
- 值越大 = 越不相似

### 方法 1：手動提供查詢向量

In [18]:
# 定義搜尋函式
def query_chromadb(query, top_n = 2):
    """
    在資料庫中搜尋相似的句子
    
    參數：
        query: 要搜尋的問題（文字）
        top_n: 要回傳幾筆最相似的結果
    
    回傳：
        (ID, 距離, 文字) 的列表
    """
    # 1. 把問題轉成向量
    query_embedding = get_embedding(query)
    
    # 2. 在資料庫中搜尋
    results = collection.query(
        query_embeddings = [query_embedding],  # 查詢向量
        n_results = top_n                       # 要回傳幾筆
    )
    
    # 3. 整理結果
    return [
        (id, score, text) 
        for id, score, text in zip(
            results['ids'][0], 
            results['distances'][0], 
            results['documents'][0]
        )
    ]

In [19]:
# 測試 1：搜尋關於食物的句子
query = "What is good food?"
results = query_chromadb(query, top_n=3)

print(f"🔍 搜尋問題：{query}\n")
print("最相似的 3 句話：\n")

for i, (id, distance, text) in enumerate(results, 1):
    print(f"{i}. 【距離：{distance:.4f}】{text}")
    print()

🔍 搜尋問題：What is good food?

最相似的 3 句話：

1. 【距離：1.0074】Food is our common ground, a universal experience.

2. 【距離：1.1380】People who love to eat are always the best people.

3. 【距離：1.1587】Let food be thy medicine and medicine be thy food.



In [20]:
# 測試 2：搜尋關於朋友的句子
query = "What is a friend?"
results = query_chromadb(query, top_n=3)

print(f"🔍 搜尋問題：{query}\n")
print("最相似的 3 句話：\n")

for i, (id, distance, text) in enumerate(results, 1):
    print(f"{i}. 【距離：{distance:.4f}】{text}")
    print()

🔍 搜尋問題：What is a friend?

最相似的 3 句話：

1. 【距離：0.9254】A true friend is someone who is there for you when he'd rather be anywhere else.

2. 【距離：1.0472】Your friend is the man who knows all about you and still likes you.

3. 【距離：1.0852】Friendship is the only cement that will ever hold the world together.



In [21]:
# 測試 3：搜尋關於自由的句子
query = "Tell me about liberty and independence"
results = query_chromadb(query, top_n=3)

print(f"🔍 搜尋問題：{query}\n")
print("最相似的 3 句話：\n")

for i, (id, distance, text) in enumerate(results, 1):
    print(f"{i}. 【距離：{distance:.4f}】{text}")
    print()

🔍 搜尋問題：Tell me about liberty and independence

最相似的 3 句話：

1. 【距離：1.0933】Liberty, when it begins to take root, is a plant of rapid growth.

2. 【距離：1.1595】Is freedom anything else than the right to live as we wish?

3. 【距離：1.1694】Life without liberty is like a body without spirit.There is nothing on this earth more to be prized than true friendship.



### 方法 2：讓 ChromaDB 自動處理向量轉換

**更方便的做法：**
- 在建立集合時就設定好「嵌入函式」
- 之後搜尋時，ChromaDB 會自動把文字轉成向量
- 你只需要輸入文字，不用手動呼叫 API

In [22]:
# 先查看目前有哪些集合
print("目前的集合：")
print(chroma_client.list_collections())

目前的集合：
[Collection(name=nice_phrases)]


In [23]:
# 刪除舊的集合（為了重新建立）
chroma_client.delete_collection("nice_phrases")
print("✅ 舊集合已刪除")

✅ 舊集合已刪除


In [26]:
import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# 建立帶有嵌入函式的集合
collection = chroma_client.create_collection(
    name = "nice_phrases",
    embedding_function = OpenAIEmbeddingFunction(
        api_key = os.getenv("OPENAI_API_KEY"),  # 明確傳入 API 金鑰
        model_name="text-embedding-3-small"     # 指定模型
    )
)

print("✅ 新集合已建立（包含自動嵌入功能）")

InternalError: Collection [nice_phrases] already exists

In [27]:
# 重新加入資料
collection.add(
    embeddings = embeddings,
    documents = phrases,
    ids = ids
)

print("✅ 資料已加入新集合")

✅ 資料已加入新集合


### 現在可以直接用文字搜尋了！

In [28]:
# 一次搜尋多個問題！
results = collection.query(
    query_texts = [
        "What is a friend?",      # 關於朋友
        "What is good food?"       # 關於食物
    ], 
    n_results = 2                  # 每個問題回傳 2 筆結果
)

# 顯示結果
print("🔍 批次搜尋結果：\n")

for i, query in enumerate(["What is a friend?", "What is good food?"]):
    print(f"問題 {i+1}：{query}")
    print("-" * 60)
    
    for j, (id, distance, text) in enumerate(zip(
        results['ids'][i], 
        results['distances'][i], 
        results['documents'][i]
    ), 1):
        print(f"  {j}. 【距離：{distance:.4f}】{text}")
    
    print()

🔍 批次搜尋結果：

問題 1：What is a friend?
------------------------------------------------------------
  1. 【距離：0.9255】A true friend is someone who is there for you when he'd rather be anywhere else.
  2. 【距離：1.0473】Your friend is the man who knows all about you and still likes you.

問題 2：What is good food?
------------------------------------------------------------
  1. 【距離：1.0074】Food is our common ground, a universal experience.
  2. 【距離：1.1379】People who love to eat are always the best people.



## 📝 總結與重點回顧

### 你學到了什麼？

1. **向量（Embedding）的概念**
   - 文字可以轉成數字（向量）
   - 意思相近的文字，向量會很接近

2. **如何使用 OpenAI Embedding API**
   - 單一文字轉換
   - 批次轉換（更有效率）

3. **向量資料庫的基本操作**
   - 建立集合（Collection）
   - 加入資料（Document + Embedding）
   - 語意搜尋（Semantic Search）

4. **ChromaDB 的兩種使用方式**
   - 手動提供向量
   - 自動嵌入函式（推薦）

### 實際應用場景

✅ **聊天機器人（Chatbot）**
- 儲存常見問題和答案
- 當用戶提問時，找出最相似的答案

✅ **文件搜尋系統**
- 公司內部知識庫
- 法律條文檢索
- 學術論文搜尋

✅ **推薦系統**
- 推薦相似的文章、商品
- 根據使用者喜好找相關內容

✅ **RAG（Retrieval-Augmented Generation）**
- 讓 AI 基於你的資料回答問題
- 避免 AI "幻覺"（亂掰答案）

### 下一步學習方向

1. **進階向量資料庫**
   - 使用持久化儲存（資料不會消失）
   - Docker 部署 ChromaDB
   - 嘗試其他資料庫（Pinecone, Weaviate）

2. **效能優化**
   - 大規模資料處理
   - 索引優化
   - 查詢速度優化

3. **整合應用**
   - 結合 LangChain 建立聊天機器人
   - 建立 RAG 問答系統
   - 多語言支援

### 常見問題 FAQ

**Q: 向量維度越高越好嗎？**
A: 不一定。維度高可以表達更多資訊，但也會：
- 增加計算成本
- 需要更多儲存空間
- 可能過擬合

OpenAI 也有 `text-embedding-3-large` 模型（3072 維），但對大部分應用來說，`small` 版本（1536 維）就夠用了。

**Q: 為什麼有時候搜尋結果不準？**
A: 可能的原因：
- 訓練資料不夠多
- 問題問得太模糊
- 領域專業詞彙（可能需要微調模型）

**Q: ChromaDB 的資料會保留嗎？**
A: 本教學使用記憶體模式，程式結束後資料就消失。實際應用中可以使用持久化模式，資料會儲存到硬碟。

**Q: 成本如何？**
A: OpenAI Embedding API 的計費方式：
- `text-embedding-3-small`: $0.02 / 1M tokens
- `text-embedding-3-large`: $0.13 / 1M tokens

一般來說非常便宜，1000 句話可能只需要幾分錢。

---

## 🎓 恭喜你完成了向量資料庫的學習！

現在你已經掌握了現代 AI 應用的核心技術之一。試著用這些知識建立自己的專案吧！